# Gemma-4 E4B QA inference (via vLLM)

Runs `google/gemma-4-E4B-it` through a local [vLLM](https://docs.vllm.ai) OpenAI-compatible server.
Gemma has no reasoning/thinking mode, so nothing needs to be disabled.

Start the server first (in a separate terminal):

```bash
vllm serve google/gemma-4-E4B-it --port 8002
```

The notebook defaults to `http://localhost:8002/v1`, but you can override it with
`VLLM_BASE_URL` and `VLLM_MODEL` in `.env`.


In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from datasets import load_from_disk
import pandas as pd

load_dotenv()

/home/faizanf/projects/LEGAL-UQA/.venv-linux/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
dataset_dict = load_from_disk("../../data/complete_dataset/")

print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['question_eng', 'question_urdu', 'context_eng', 'context_urdu', 'answer_eng', 'answer_urdu', 'context_index', '__index_level_0__'],
        num_rows: 495
    })
    validation: Dataset({
        features: ['question_eng', 'question_urdu', 'context_eng', 'context_urdu', 'answer_eng', 'answer_urdu', 'context_index', '__index_level_0__'],
        num_rows: 124
    })
})


In [3]:
validation_df = pd.DataFrame(dataset_dict['validation'])
df = validation_df[['question_urdu', 'answer_urdu']].copy()
df['id'] = range(len(df))
df.rename(columns={'question_urdu': 'question', 'answer_urdu': 'answers'}, inplace=True)
df = df[['id', 'question', 'answers']]

display(df)


,id,answers
0,0,اسلامی کونسل کا کوئی رکن تین سال کی مدت کے لیے...
1,1,پاکستان میں یوم آغاز سے قبل قبائلی علاقے جات م...
2,2,نہیں، کسی عدالت عالیہ کے جج کا اس کی رضامندی ک...
3,3,وفاقی حکومت یا، جیسی بھی صورت ہو، صوبائی حکومت...
4,4,کسی رکن کو منحرف قرار دینے سے پہلے، پارٹی کے س...
...,...,...
119,119,وزیر اعظم مشترکہ مفادات کی کونسل کا چیئرمین ہو...
120,120,نہیں، کسی شخص کو شہادت حاصل کرنے کی غرض سے اذی...
121,121,سینیٹ میں اسپیکر اور ڈپٹی اسپیکر کے مساوی عہدے...
122,122,مالی بل کی ابتدا قومی اسمبلی میں ہوگی۔


In [4]:
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8002/v1").rstrip("/")
VLLM_MODEL = os.getenv("VLLM_MODEL", "google/gemma-4-E4B-it")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "EMPTY")

# Reuse one client for the full notebook run.
client = OpenAI(api_key=VLLM_API_KEY, base_url=VLLM_BASE_URL)


def get_urdu_response(context, question):
    instruction = "You are a Pakistani legal expert who speaks in Urdu. You only respond in Urdu script."
    prompt = f"""Below is a context provided, followed by a question. Your task is to answer the question based on the given context. The answer should consist only of the response to the question in Urdu and should not include any other information or commentary.

    Context: ```{context}```

    Question: ```{question}```

    Answer:"""

    completion = client.chat.completions.create(
        model=VLLM_MODEL,
        temperature=0.0,
        max_tokens=1024,
        messages=[
            {"role": "user", "content": f"{instruction}\n\n{prompt}"},
        ],
    )

    return completion.choices[0].message.content


In [5]:
from tqdm import tqdm

predicted_answers = []

for i in tqdm(range(len(dataset_dict['validation'])), desc="Processing responses", unit="item"):
    test_point = dataset_dict['validation'][i]

    context = test_point['context_urdu']
    question = test_point['question_urdu']

    predicted_answer = get_urdu_response(context=context, question=question)

    predicted_answers.append(predicted_answer)

Processing responses:   0%|          | 0/124 [00:00<?, ?item/s]

Processing responses:   1%|          | 1/124 [00:00<01:37,  1.26item/s]

Processing responses:   2%|▏         | 2/124 [00:01<01:20,  1.52item/s]

Processing responses:   2%|▏         | 3/124 [00:01<01:05,  1.85item/s]

Processing responses:   3%|▎         | 4/124 [00:02<00:51,  2.33item/s]

Processing responses:   4%|▍         | 5/124 [00:02<00:57,  2.06item/s]

... (112 outputs left out)

Processing responses:  98%|█████████▊| 122/124 [01:12<00:00,  2.71item/s]

Processing responses:  99%|█████████▉| 123/124 [01:13<00:00,  3.10item/s]

Processing responses: 100%|██████████| 124/124 [01:13<00:00,  2.90item/s]

Processing responses: 100%|██████████| 124/124 [01:13<00:00,  1.68item/s]

In [6]:
predicted_answers[:5]

['اسلامی کونسل کا کوئی رکن تین سال کی مدت کے لیے اپنے عہدے پر فائز رہے گا۔',
 'بلوچستان اور خیبر پختونخوا کے قبائلی علاقے جات، امب، چترال، دیر اور سوات کی سابقہ ریاستیں۔',
 'نہیں، کسی جج کا اس طرح تبادلہ نہیں کیا جائے گا بجز اس کی رضامندی سے۔',
 'وفاقی حکومت یا صوبائی حکومت وقتاً فوقتاً ترمیم کر سکتی ہے۔',
 'اعلان کرنے سے پہلے، پارٹی کا سربراہ مذکورہ رکن کو اس بارے میں اظہار وجوہ کا موقع فراہم کرے گا کہ کیوں نہ اس کے خلاف مذکورہ اعلان کر دیا جائے۔']

In [7]:
df['requested_model'] = 'gemma-4-e4b'
df['model_version'] = VLLM_MODEL
df['reasoning_mode'] = 'none'
df['prediction_text_gemma_4_e4b'] = predicted_answers
df = df[[
    'id',
    'question',
    'answers',
    'requested_model',
    'model_version',
    'reasoning_mode',
    'prediction_text_gemma_4_e4b',
]]


In [8]:
display(df.head())

,id,answers,prediction_text_gemma_4_e4b
0,0,اسلامی کونسل کا کوئی رکن تین سال کی مدت کے لیے...,اسلامی کونسل کا کوئی رکن تین سال کی مدت کے لیے...
1,1,پاکستان میں یوم آغاز سے قبل قبائلی علاقے جات م...,بلوچستان اور خیبر پختونخوا کے قبائلی علاقے جات...
2,2,نہیں، کسی عدالت عالیہ کے جج کا اس کی رضامندی ک...,نہیں، کسی جج کا اس طرح تبادلہ نہیں کیا جائے گا...
3,3,وفاقی حکومت یا، جیسی بھی صورت ہو، صوبائی حکومت...,وفاقی حکومت یا صوبائی حکومت وقتاً فوقتاً ترمیم...
4,4,کسی رکن کو منحرف قرار دینے سے پہلے، پارٹی کے س...,اعلان کرنے سے پہلے، پارٹی کا سربراہ مذکورہ رکن...


In [9]:
df.to_csv('../../data/model_qa_responses/gemma-4-e4b-responses.csv', index=False)
